# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

**Data source:** the ranked queue below is built from `work/outputs/w05_model_test_predictions.csv` —
the Week-5 model's scores on its **held-out test clients** (the client-grouped split audited honestly in
`w06_validation_audit.ipynb`, precision@50 = 0.76 vs a 0.42 baseline-rule and a 0.52 base rate). Using the
held-out set (not the training rows) means every row in this queue is a page the model scored **without
having seen that client during training** — the closest honest proxy this project has for "a new page
arriving today."

**Reason codes** are built only from feature columns (impressions, CTR, position, freshness, age) — never
from the label — so the logic mirrors what would be available at real prediction time. Each page gets one
primary reason code (first rule that matches, in priority order) and a mapped **archetype -> action**.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "work/outputs/w05_model_test_predictions.csv").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

PRED_PATH = ROOT / "work/outputs/w05_model_test_predictions.csv"
queue = pd.read_csv(PRED_PATH)

print(f"rows (held-out test clients only) = {len(queue):,}")
print(f"columns available for reason codes: {[c for c in queue.columns if c not in ('is_declining_label', 'trend_direction')]}")

# --- Archetype / reason-code / action rules -------------------------------
# Priority order matters: first matching rule wins. Thresholds reuse the
# Week-4 baseline rule's definitions of "visible" and "reachable position"
# so the playbook stays consistent with earlier, already-reviewed logic.
VISIBLE_MIN, VISIBLE_MAX = 300, 30000
REACHABLE_MIN, REACHABLE_MAX = 3, 20
LOW_CTR = 0.5
STALE_DAYS = 91

def visible(row):
    return VISIBLE_MIN <= row["impressions_90d"] < VISIBLE_MAX

def reachable(row):
    return REACHABLE_MIN < row["avg_position"] <= REACHABLE_MAX

def classify(row):
    is_visible = visible(row)
    is_reachable = reachable(row)
    is_low_ctr = row["ctr"] < LOW_CTR
    is_stale = row["days_since_last_update"] >= STALE_DAYS
    is_aging = row["age_tier"] in ("181-365", "365+")

    if is_visible and is_reachable and is_low_ctr:
        return (
            "LOW_CTR_VISIBLE",
            "Visible + reachable position, weak CTR",
            "Rewrite title/meta and snippet",
            "low",
        )
    if is_visible and is_stale:
        return (
            "STALE_UPDATE_DUE",
            "Visible, no update in 91+ days",
            "Refresh and expand content",
            "medium",
        )
    if is_aging and is_reachable:
        return (
            "AGING_REACHABLE",
            "Older content still in a reachable position",
            "Schedule editorial review before the decay window",
            "medium",
        )
    if not is_visible:
        return (
            "LOW_VISIBILITY",
            "Below the visible-impressions band",
            "Monitor only -- not enough traffic to justify effort yet",
            "none",
        )
    return (
        "WATCH_LIST",
        "No high-confidence rule matched",
        "No action yet -- add to watch list",
        "none",
    )

classified = queue.apply(classify, axis=1, result_type="expand")
classified.columns = ["reason_code", "reason_detail", "recommended_action", "effort_tier"]
queue = pd.concat([queue, classified], axis=1)

# Value tier: relative opportunity size, independent of the effort needed
queue["value_tier"] = pd.qcut(
    queue["impressions_90d"], q=[0, 0.5, 0.85, 1.0], labels=["low", "medium", "high"]
)

queue_ranked = queue.sort_values("model_score", ascending=False).reset_index(drop=True)
queue_ranked.insert(0, "queue_rank", np.arange(1, len(queue_ranked) + 1))

display_cols = [
    "queue_rank", "content_id", "content_type", "model_score", "reason_code",
    "recommended_action", "effort_tier", "value_tier", "avg_position", "ctr",
    "days_since_last_update",
]
print("\nTop 15 ranked actions:")
print(queue_ranked[display_cols].head(15).to_string(index=False, float_format=lambda x: f"{x:.3f}"))

print("\nReason code distribution across the full held-out queue:")
print(queue_ranked["reason_code"].value_counts())


rows (held-out test clients only) = 7,115
columns available for reason codes: ['content_id', 'content_type', 'impression_tier', 'position_tier', 'age_tier', 'freshness_tier', 'impressions_90d', 'ctr', 'avg_position', 'days_since_last_update', 'model_score', 'baseline_score', 'model_rank', 'pred_at_050', 'error_type']



Top 15 ranked actions:
 queue_rank           content_id       content_type  model_score     reason_code                                       recommended_action effort_tier value_tier  avg_position   ctr  days_since_last_update
          1 content_a8864e189b2e comparison article        0.966  LOW_VISIBILITY Monitor only -- not enough traffic to justify effort yet        none        low         6.400 0.000                       8
          2 content_a928cb66d230    keyword article        0.963  LOW_VISIBILITY Monitor only -- not enough traffic to justify effort yet        none        low         4.200 0.000                      20
          3 content_7be5f150dc65    keyword article        0.953  LOW_VISIBILITY Monitor only -- not enough traffic to justify effort yet        none        low         5.900 0.000                      20
          4 content_5d77d3077984    keyword article        0.949  LOW_VISIBILITY Monitor only -- not enough traffic to justify effort yet        none       

**A note on what "top of the queue" means:** sorting purely by `model_score` (probability of
decline) surfaces pages that are *already* nearly dead — very low traffic, 0% CTR — as the top hits. That's
technically correct (they are the most confidently "declining"), but there's often nothing left to recover,
so `LOW_VISIBILITY` correctly routes them to "monitor only" rather than an effort-spending action. The queue
a reviewer should actually work top-down is the **actionable priority queue** below: real recommended
actions (`effort_tier != "none"`), ordered by opportunity size first, then by how confident the model is.
This is the cost/value step — spend review time where there is both a real action and real traffic to
recover, not just where the model is most confident something is wrong.

In [2]:
actionable = queue_ranked[queue_ranked["effort_tier"] != "none"].copy()
value_order = pd.Categorical(actionable["value_tier"], categories=["high", "medium", "low"], ordered=True)
actionable = actionable.assign(_value_order=value_order).sort_values(
    ["_value_order", "model_score"], ascending=[True, False]
).drop(columns="_value_order").reset_index(drop=True)
actionable.insert(0, "priority_rank", np.arange(1, len(actionable) + 1))

print(f"actionable rows (effort_tier != 'none'): {len(actionable):,} of {len(queue_ranked):,} "
      f"({len(actionable) / len(queue_ranked):.1%})")
print("\nTop 15 by actionable priority (value tier first, then model confidence):")
print(actionable[["priority_rank", "content_id", "content_type", "model_score", "reason_code",
      "recommended_action", "effort_tier", "value_tier"]].head(15).to_string(
    index=False, float_format=lambda x: f"{x:.3f}"
))

queue_ranked["is_actionable"] = queue_ranked["effort_tier"] != "none"


actionable rows (effort_tier != 'none'): 3,859 of 7,115 (54.2%)

Top 15 by actionable priority (value tier first, then model confidence):
 priority_rank           content_id       content_type  model_score      reason_code                                recommended_action effort_tier value_tier
             1 content_c82bc0c24241    keyword article        0.927  LOW_CTR_VISIBLE                    Rewrite title/meta and snippet         low       high
             2 content_d6e1bbb4a996    keyword article        0.916  LOW_CTR_VISIBLE                    Rewrite title/meta and snippet         low       high
             3 content_66458ac1b739    keyword article        0.912 STALE_UPDATE_DUE                        Refresh and expand content      medium       high
             4 content_11a4f985f14d    keyword article        0.910  LOW_CTR_VISIBLE                    Rewrite title/meta and snippet         low       high
             5 content_5d5653c4eb4f    keyword article        0.897  LOW

### The decay/refresh insight, applied to the queue

Two things line up here that are worth naming explicitly, because they justify *why* `STALE_UPDATE_DUE`
and `AGING_REACHABLE` exist as separate reason codes rather than one generic "old content" bucket:

- **The research paper's Finding #2** (page 7) shows FlyRank's portfolio-wide health score peaking at
  61-90 days and hitting a "decay cliff" at 271-365 days — the paper frames this as a window to catch
  content *before* it falls off, not after.
- **The research paper's Finding #4** (page 9) shows refreshing 365+ day content associated with a
  measured 3.2x health boost and 57x impression difference in this portfolio (with the caveat, which the
  paper itself raises, that the 361+ bucket is a small, unstable sample).
- **My own Section-1 sanity check in `w06_validation_audit.ipynb`** showed the same directional shape on
  this dataset using `is_declining_label` instead of health score: the `91-180` freshness tier had the
  highest decline rate (0.611) of any freshness bucket in my data.

So the `days_since_last_update >= 91` threshold used for `STALE_UPDATE_DUE` above is not arbitrary — it
sits right at the age where both the paper's aggregate curve and my own model's behavior start showing
more decline. The honest framing: this is a **directional, decision-support signal for prioritizing a
review queue**, not a guarantee that refreshing any specific page will reproduce the paper's 3.2x/57x
figures, which come from a different (and in the 365+ case, small) sample.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

## Intended use

This playbook is a **ranked review queue** for a content strategist or SEO reviewer who needs to decide
*which pages to look at first* out of a large portfolio. It is meant to sit in front of a human — a
prioritization aid, not a publishing pipeline.

**Who uses it:** a content reviewer or strategist doing routine triage (e.g., "what should I look at this
sprint?").

**What it is for:** ranking and reason-coding pages worth a closer look, and grouping them into archetypes
so similar pages get a similar first pass.

**What it is explicitly NOT for:** deciding what to publish, automatically editing content, or making
client-facing claims about guaranteed traffic/ranking outcomes.

## Where it stops being valid

- **Client generalization is measured, not assumed.** The model's honest precision@50 on held-out clients
  is 0.76 (vs a 0.52 base rate and a 0.42 rule-based baseline, from `w06_validation_audit.ipynb`) — useful,
  but well short of 1.0. Roughly 1 in 4 of the top-50 ranked pages will not actually be declining.
- **Single snapshot, single portfolio.** The training data is a 90-day trailing window from one set of
  clients; nothing here has been tested against a different vertical, a different search environment, or a
  future time period.
- **Reason codes are rule-based on top of a probabilistic score**, so a page can have a plausible reason
  code and still be a false positive — the reason code explains *why the queue flagged it*, not proof that
  the recommended action will work.
- **content_age_days / days_since_last_update are snapshot-time features** (per the Section-3 leakage audit
  in w06) — if this were ever wired to a live feed, those columns would need to be recomputed relative to
  the actual review date, not frozen at the original data pull.

In [3]:
# Coverage + honest-limit numbers this playbook actually rests on (traced back to
# work/outputs/w05_model_metrics.json so the claims above are not just prose).
import json

metrics_path = ROOT / "work/outputs/w05_model_metrics.json"
w05_metrics = json.loads(metrics_path.read_text())

model_row = next(r for r in w05_metrics["comparison"] if r["method"] == "logistic_regression")
baseline_row = next(r for r in w05_metrics["comparison"] if r["method"] == "week4_baseline_rule")

print(f"held-out test clients: {w05_metrics['split']['test_clients']} "
      f"(of {w05_metrics['split']['train_clients'] + w05_metrics['split']['test_clients']} total)")
print(f"test base rate: {w05_metrics['split']['test_base_rate']:.3f}")
print(f"model precision@50 (honest, grouped split): {model_row['precision_at_50']:.3f}")
print(f"rule-based baseline precision@50: {baseline_row['precision_at_50']:.3f}")
print(f"expected false-positive share in top 50: {1 - model_row['precision_at_50']:.0%} "
      "-- this is why review stays mandatory, not optional.")


held-out test clients: 8 (of 32 total)
test base rate: 0.517
model precision@50 (honest, grouped split): 0.760
rule-based baseline precision@50: 0.420
expected false-positive share in top 50: 24% -- this is why review stays mandatory, not optional.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

## Human review before acting

Nothing in this queue should be actioned without a person checking:

- **Content quality and accuracy** — does the page still say something true and useful, independent of
  its metrics?
- **Brand voice and tone** — does a rewrite of title/meta/snippet still sound like the client?
- **Context the model can't see** — a seasonal page, a page tied to a discontinued product, or a page
  under legal/compliance review can look "stale" or "declining" for reasons that have nothing to do with
  SEO opportunity.
- **Archetype fit** — does the reason code actually match what a human sees on the page, or is this one of
  the ~24% of top-50 cases where the model's flag doesn't hold up?

## What should NEVER be automated

- **No auto-publishing.** Rewritten titles, meta descriptions, or content changes go through a human editor
  before going live.
- **No auto-deletion or deindexing.** A page scored as low-value or "WATCH_LIST" is a candidate for review,
  never for automatic removal.
- **No client-facing automated claims.** Nothing from this queue (a reason code, a value tier, a score)
  should be sent to a client as a guarantee or a report line without a human rewriting it in
  decision-support language.
- **No bulk unattended edits.** Even a "low effort" action (title/meta rewrite) is applied page-by-page with
  a person reviewing the diff, not scripted across the whole queue at once.
- **No YMYL / high-stakes content on autopilot.** Content in sensitive categories (health, finance, legal,
  safety) always gets senior human review regardless of reason code or model score.

In [4]:
# Flag content that needs escalated (senior) review rather than routine review,
# based on content_type -- a simple, auditable proxy since this dataset has no
# explicit YMYL/compliance flag column.
sensitive_keywords = ("guide", "comparison")  # illustrative: content types worth a second look
queue_ranked["needs_senior_review"] = queue_ranked["content_type"].str.contains(
    "|".join(sensitive_keywords), case=False, na=False
)

print("Content types present in the queue:")
print(queue_ranked["content_type"].value_counts())

print("\nRows flagged for senior review (illustrative content-type proxy):")
print(f"  {queue_ranked['needs_senior_review'].sum():,} of {len(queue_ranked):,} rows "
      f"({queue_ranked['needs_senior_review'].mean():.1%})")
print("\nNote: this is a placeholder proxy, not a real compliance classifier. In a real")
print("deployment, the no-go / escalation list should be driven by an actual content-category")
print("or compliance tag, not a keyword match on content_type.")


Content types present in the queue:
content_type
keyword article       6418
comparison article     697
Name: count, dtype: int64

Rows flagged for senior review (illustrative content-type proxy):
  697 of 7,115 rows (9.8%)

Note: this is a placeholder proxy, not a real compliance classifier. In a real
deployment, the no-go / escalation list should be driven by an actual content-category
or compliance tag, not a keyword match on content_type.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

## Monitoring and retrain triggers

The queue is only as good as the assumption that new pages look like the pages the model was trained and
tested on. These are the light-weight signals that would tell a reviewer the recommendations have gone
stale — none of this is an automated production monitoring system, just the checks a human running this
periodically should look at.

| Trigger | Threshold (illustrative) | Why |
|---|---|---|
| Precision@50 on newly labeled pages | drops below 0.60 (well above the 0.42 rule-baseline floor, comfortably below the current 0.76) | signals the model is no longer beating the simple rule by a meaningful margin |
| Portfolio base rate shift | moves more than ~10 points from the 0.52 base rate seen in Week 5/6 | the model was tuned around this base rate; a big shift changes what "good" precision even means |
| Reason-code mix shift | `WATCH_LIST` share grows past ~25% of the queue (currently a minority bucket) | means the existing rules are matching fewer pages -- the feature distribution has likely moved |
| New client onboarding | any time | the model has only been validated on held-out *existing* clients, never on a genuinely new client — treat early recommendations for a new client with extra caution until enough labeled outcomes accumulate |
| Time since last refit | ~1 quarter (matches the paper's 90-day rolling window) | keeps the training window aligned with how the underlying features (90-day trailing metrics) are defined |

**Retrain, not just re-score, when:** precision@50 trigger fires on two consecutive review cycles, or a
new client segment is added that looks structurally different from the training portfolio (e.g., a new
`content_type` or a very different age/freshness distribution).

In [5]:
# Illustrative trigger check against the current honest metrics -- this is a check
# a reviewer would re-run each cycle with fresh numbers, not a live monitor.
precision_trigger = 0.60
base_rate_reference = w05_metrics["split"]["test_base_rate"]
base_rate_drift_limit = 0.10

current_precision = model_row["precision_at_50"]
current_base_rate = w05_metrics["split"]["test_base_rate"]  # same eval run; would be a fresh number in practice
watch_list_share = (queue_ranked["reason_code"] == "WATCH_LIST").mean()

print(f"precision@50 = {current_precision:.3f} -> "
      f"{'TRIGGER: below 0.60, review before trusting the queue' if current_precision < precision_trigger else 'ok, above trigger'}")
print(f"base rate drift vs reference ({base_rate_reference:.3f}) = "
      f"{abs(current_base_rate - base_rate_reference):.3f} -> "
      f"{'TRIGGER' if abs(current_base_rate - base_rate_reference) > base_rate_drift_limit else 'ok, within range'}")
print(f"WATCH_LIST share = {watch_list_share:.1%} -> "
      f"{'TRIGGER: above 25%, rules may be stale' if watch_list_share > 0.25 else 'ok, below 25%'}")


precision@50 = 0.760 -> ok, above trigger
base rate drift vs reference (0.517) = 0.000 -> ok, within range
WATCH_LIST share = 11.6% -> ok, below 25%


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [6]:
import json

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

OUTPUT_DIR = ROOT / "work/outputs"
FIGURE_DIR = ROOT / "work/figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# 1) Ranked queue CSV -- regenerated each run, intentionally kept out of git (work/**/*.csv)
queue_path = OUTPUT_DIR / "w07_action_queue.csv"
export_cols = [
    "queue_rank", "content_id", "content_type", "model_score", "reason_code",
    "reason_detail", "recommended_action", "effort_tier", "value_tier",
    "is_actionable", "needs_senior_review", "impressions_90d", "avg_position", "ctr",
    "days_since_last_update", "age_tier", "freshness_tier",
]
queue_ranked[export_cols].to_csv(queue_path, index=False)
print(f"wrote ranked queue: {queue_path} ({len(queue_ranked):,} rows)")

# 2) Summary JSON -- committed, this is the "receipt" the paper's numbers trace back to
summary = {
    "source_predictions": "work/outputs/w05_model_test_predictions.csv",
    "n_queue_rows": int(len(queue_ranked)),
    "n_actionable_rows": int(queue_ranked["is_actionable"].sum()),
    "model_precision_at_50": float(model_row["precision_at_50"]),
    "baseline_precision_at_50": float(baseline_row["precision_at_50"]),
    "test_base_rate": float(w05_metrics["split"]["test_base_rate"]),
    "reason_code_counts": queue_ranked["reason_code"].value_counts().to_dict(),
    "effort_tier_counts": queue_ranked["effort_tier"].value_counts().to_dict(),
    "value_tier_counts": queue_ranked["value_tier"].value_counts().to_dict(),
    "needs_senior_review_rows": int(queue_ranked["needs_senior_review"].sum()),
    "monitoring_triggers": {
        "precision_at_50_floor": precision_trigger,
        "base_rate_drift_limit": base_rate_drift_limit,
        "watch_list_share_limit": 0.25,
    },
    "notes": "Decision-support queue for human-reviewed content triage. Not a production automation.",
}
summary_path = OUTPUT_DIR / "w07_playbook_summary.json"
summary_path.write_text(json.dumps(summary, indent=2))
print(f"wrote summary: {summary_path}")

# 3) Figures -- committed, reused by the paper next week
fig1, ax1 = plt.subplots(figsize=(7, 4))
reason_counts = queue_ranked["reason_code"].value_counts().sort_values()
ax1.barh(reason_counts.index, reason_counts.values, color="#3b6ea5")
ax1.set_xlabel("pages in held-out queue")
ax1.set_title("Reason code distribution (held-out test clients)")
fig1.tight_layout()
fig1_path = FIGURE_DIR / "w07_reason_code_distribution.png"
fig1.savefig(fig1_path, dpi=150)
plt.close(fig1)
print(f"wrote figure: {fig1_path}")

fig2, ax2 = plt.subplots(figsize=(6, 4))
methods = ["baseline rule", "model (this playbook)"]
values = [baseline_row["precision_at_50"], model_row["precision_at_50"]]
bars = ax2.bar(methods, values, color=["#a5a5a5", "#3b6ea5"])
ax2.axhline(w05_metrics["split"]["test_base_rate"], color="black", linestyle="--", linewidth=1,
            label=f"base rate ({w05_metrics['split']['test_base_rate']:.2f})")
ax2.set_ylabel("precision@50 (held-out clients)")
ax2.set_ylim(0, 1)
ax2.set_title("Ranking quality behind this playbook's queue")
ax2.legend()
for bar, v in zip(bars, values):
    ax2.text(bar.get_x() + bar.get_width() / 2, v + 0.02, f"{v:.2f}", ha="center")
fig2.tight_layout()
fig2_path = FIGURE_DIR / "w07_precision_at_50_comparison.png"
fig2.savefig(fig2_path, dpi=150)
plt.close(fig2)
print(f"wrote figure: {fig2_path}")

print("\nExports ready for the paper: queue CSV (regenerated, git-ignored), summary JSON (committed),")
print("two figures under work/figures/ (committed).")


wrote ranked queue: E:\coding ground\FLR-ML-01\work\outputs\w07_action_queue.csv (7,115 rows)
wrote summary: E:\coding ground\FLR-ML-01\work\outputs\w07_playbook_summary.json
wrote figure: E:\coding ground\FLR-ML-01\work\figures\w07_reason_code_distribution.png


wrote figure: E:\coding ground\FLR-ML-01\work\figures\w07_precision_at_50_comparison.png

Exports ready for the paper: queue CSV (regenerated, git-ignored), summary JSON (committed),
two figures under work/figures/ (committed).


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.